### SelfQueryRetriever

In [2]:
!uv --version

uv 0.12.17 (635500036 2026-09-18 x86_64-pc-windows-msvc)


In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain_teddynote import logging

logging.langsmith("test0922")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0922


In [5]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

docs = [
    Document(
        page_content="수분 가득한 히알루론산 세럼으로 피부 속 깊은 곳까지 수분을 공급합니다.",
        metadata={"year": 2024, "category": "스킨케어", "user_rating": 4.7},
    ),
    Document(
        page_content="24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.",
        metadata={"year": 2023, "category": "메이크업", "user_rating": 4.5},
    ),
    Document(
        page_content="식물성 성분으로 만든 저자극 클렌징 오일, 메이크업과 노폐물을 부드럽게 제거합니다.",
        metadata={"year": 2023, "category": "클렌징", "user_rating": 4.8},
    ),
    Document(
        page_content="비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.",
        metadata={"year": 2023, "category": "스킨케어", "user_rating": 4.6},
    ),
    Document(
        page_content="롱래스팅 립스틱, 선명한 발색과 촉촉한 사용감으로 하루종일 편안하게 사용 가능합니다.",
        metadata={"year": 2024, "category": "메이크업", "user_rating": 4.4},
    ),
    Document(
        page_content="자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.",
        metadata={"year": 2024, "category": "선케어", "user_rating": 4.9},
    ),
]

vectorstore = Chroma.from_documents(
    docs, OpenAIEmbeddings(model="text-embedding-3-small")
)

In [6]:
from langchain_classic.chains.query_constructor.base import AttributeInfo

metadata_field_info = [
    AttributeInfo(
        name="category",
        description="The category of the cosmetic product. One of ['스킨케어', '메이크업', '클렌징', '선케어']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the cosmetic product was released",
        type="integer",
    ),
    AttributeInfo(
        name="user_rating",
        description="A user rating for the cosmetic product, ranging from 1 to 5",
        type="float",
    ),
]

In [7]:
# !uv add lark

In [ ]:
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_openai import ChatOpenAI
from langchain_community.query_constructors.chroma import ChromaTranslator

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="Brief summary of a cosmetic product",
    metadata_field_info=metadata_field_info,
    structured_query_translator=ChromaTranslator(), # 버그있는 자동 감지 로직 우회
)

C:\Users\user\AppData\Local\Temp\ipykernel_10008\31542067.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.query_constructors.chroma import ChromaTranslator


In [9]:
retriever.invoke("평점이 4.8 이상인 제품을 추천해줘")

[Document(id='1f577ba8-18bd-4b1c-9c53-077e926ef163', metadata={'year': 2024, 'category': '선케어', 'user_rating': 4.9}, page_content='자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.'),
 Document(id='c280f86d-ef48-46d5-bb96-0e7fe431570a', metadata={'user_rating': 4.8, 'year': 2023, 'category': '클렌징'}, page_content='식물성 성분으로 만든 저자극 클렌징 오일, 메이크업과 노폐물을 부드럽게 제거합니다.')]

In [10]:
retriever.invoke("2023년에 출시된 상품을 추춴해줘")

[Document(id='1e644bf8-8f18-4be8-bfe6-51e23c2ff4d9', metadata={'category': '메이크업', 'year': 2023, 'user_rating': 4.5}, page_content='24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.'),
 Document(id='acc52870-cb4c-44d5-a451-e859b054809a', metadata={'year': 2023, 'user_rating': 4.6, 'category': '스킨케어'}, page_content='비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.'),
 Document(id='c280f86d-ef48-46d5-bb96-0e7fe431570a', metadata={'category': '클렌징', 'year': 2023, 'user_rating': 4.8}, page_content='식물성 성분으로 만든 저자극 클렌징 오일, 메이크업과 노폐물을 부드럽게 제거합니다.')]

In [11]:
retriever.invoke("카테고리가 선케어인 상품을 추천해줘")

[Document(id='1f577ba8-18bd-4b1c-9c53-077e926ef163', metadata={'year': 2024, 'user_rating': 4.9, 'category': '선케어'}, page_content='자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.')]

In [12]:
retriever.invoke(
    "카테고리가 메이크업인 상품 중에서 평점이 4.5 이상인 상품을 추천해줘"
)

[Document(id='1e644bf8-8f18-4be8-bfe6-51e23c2ff4d9', metadata={'user_rating': 4.5, 'category': '메이크업', 'year': 2023}, page_content='24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.')]

In [14]:
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="Brief summary of a cosmetic product",
    metadata_field_info=metadata_field_info,
    enable_limit=True,
    search_kwargs={"k": 2},
    structured_query_translator=ChromaTranslator(),
)

In [15]:
retriever.invoke("2023년에 출시된 상품을 추천해줘")

[Document(id='1e644bf8-8f18-4be8-bfe6-51e23c2ff4d9', metadata={'category': '메이크업', 'user_rating': 4.5, 'year': 2023}, page_content='24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.'),
 Document(id='acc52870-cb4c-44d5-a451-e859b054809a', metadata={'user_rating': 4.6, 'year': 2023, 'category': '스킨케어'}, page_content='비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.')]

In [16]:
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="Brief summary of a cosmetic product",
    metadata_field_info=metadata_field_info,
    enable_limit=True,
    structured_query_translator=ChromaTranslator(),
)

In [19]:
retriever.invoke("2023년에 출시된 상품 1개를 추천해줘")

[Document(id='1e644bf8-8f18-4be8-bfe6-51e23c2ff4d9', metadata={'user_rating': 4.5, 'year': 2023, 'category': '메이크업'}, page_content='24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.')]

In [20]:
retriever.invoke("2023년에 출시된 상품 2개를 추천해줘")

[Document(id='1e644bf8-8f18-4be8-bfe6-51e23c2ff4d9', metadata={'user_rating': 4.5, 'year': 2023, 'category': '메이크업'}, page_content='24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.'),
 Document(id='acc52870-cb4c-44d5-a451-e859b054809a', metadata={'year': 2023, 'user_rating': 4.6, 'category': '스킨케어'}, page_content='비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.')]

In [21]:
from langchain_classic.chains.query_constructor.base import (
    StructuredQueryOutputParser,
    get_query_constructor_prompt,
)

prompt = get_query_constructor_prompt(
    "Brief summary of a cosmetic product",
    metadata_field_info,
)

output_parser = StructuredQueryOutputParser.from_components()

query_constructor_chain = prompt | llm | output_parser

In [22]:
print(prompt.format(query="dummy question"))

Your goal is to structure the user's query to match the request schema provided below.

<< Structured Request Schema >>
When responding use a markdown code snippet with a JSON object formatted in the following schema:

```json
{
    "query": string \ text string to compare to document contents
    "filter": string \ logical condition statement for filtering documents
}
```

The query string should contain only text that is expected to match the contents of documents. Any conditions in the filter should not be mentioned in the query as well.

A logical condition statement is composed of one or more comparison and logical operation statements.

A comparison statement takes the form: `comp(attr, val)`:
- `comp` (eq | ne | gt | gte | lt | lte | contain | like | in | nin): comparator
- `attr` (string):  name of attribute to apply the comparison to
- `val` (string): is the comparison value

A logical operation statement takes the form `op(statement1, statement2, ...)`:
- `op` (and | or | not

In [25]:
query_output = query_constructor_chain.invoke(
    {
        "query": "2023년도에 출시한 상품 중 평점이 4.5 이상인 상품중에서 스킨케어 제품을 추천해줘"
    }
)

In [26]:
query_output.filter.arguments

[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='year', value=2023),
 Comparison(comparator=<Comparator.GTE: 'gte'>, attribute='user_rating', value=4.5),
 Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='category', value='스킨케어')]

In [27]:
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator

retriever = SelfQueryRetriever(
    query_constructor=query_constructor_chain,
    vectorstore=vectorstore,
    structured_query_translator=ChromaTranslator(),
)

In [28]:
retriever.invoke(
    "2023년도에 출시한 상품 중 평점이 4.5 이상인 상품중에서 스킨케어 제품을 추천해줘"
)

[Document(id='acc52870-cb4c-44d5-a451-e859b054809a', metadata={'category': '스킨케어', 'year': 2023, 'user_rating': 4.6}, page_content='비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.')]